# 워크숍 1 - 기초

이 노트북에서는 Menlo Robotics 플랫폼에 연결하고, 시뮬레이션 로봇을 만든 뒤, **menlo_robot_sdk**를 사용해 이동 명령을 보내는 과정을 단계별로 배웁니다.

이 튜토리얼을 마치면 다음을 할 수 있습니다.
1. API 키를 사용해 Menlo 플랫폼에 인증하기
2. 가상 로봇을 만들고 연결하기
3. 브라우저에서 로봇을 볼 수 있는 3D 뷰어 열기
4. 로봇이 사용할 수 있는 스킬 확인하기
5. `set_velocity`와 `go_to` 명령으로 로봇 움직이기
6. 로봇의 1인칭 카메라 이미지 캡처하기
7. 장애물을 돌아가는 간단한 명령 체인 작성하기


## 준비 사항

시작하기 전에 다음을 준비하세요.

- **Google 계정**(Google Colab 사용)
- Menlo API 키([platform.menlo.ai](http://platform.menlo.ai/) -> Settings -> API Keys에서 발급)
- **Google Chrome**(3D 뷰어는 Chrome에서 가장 안정적으로 동작합니다)


## API 키 설정하기

이 노트북은 두 가지 방식을 지원합니다. 여러분의 환경에 맞는 방식을 선택하세요.

### 모드 A: Google Colab(Secrets 관리자)

Colab의 내장 Secrets 관리자는 키를 안전하게 저장하고 세션이 바뀌어도 유지합니다.

1. Colab 왼쪽 사이드바에서 **키 아이콘**을 클릭합니다.
2. **"Add new secret"**을 클릭하고 아래 키를 각각 추가합니다.

| Secret 이름 | 값 |
|---|---|
| `MENLO_API_KEY` | Menlo API 키([platform.menlo.ai](http://platform.menlo.ai/) -> Settings -> API Keys) |
| `TOKAMAK_API_KEY` | Tokamak API 키(섹션 7의 VLM에서 사용) |

3. 각 secret의 **"Notebook access"**를 켭니다.

> **참고:** 노트북을 공유해도 secret은 포함되지 않습니다. 각 사용자가 자신의 키를 직접 설정해야 합니다.

### 모드 B: 로컬 / `.env` 파일

VS Code, JupyterLab 같은 로컬 환경에서 실행한다면 이 노트북과 같은 폴더에 `.env` 파일을 만드세요.

```
MENLO_API_KEY=sk_live_abc123...
TOKAMAK_API_KEY=tok_abc123...
```

다음 셀은 현재 환경을 자동으로 감지하고 알맞은 방식으로 키를 불러옵니다.


---

# 섹션 4: SDK 기본 사용법


## SDK의 주요 import

`menlo_robot_sdk` 패키지는 몇 가지 핵심 구성 요소를 제공합니다.

| Import | 역할 |
|---|---|
| `AsyncClient` | Menlo 플랫폼과 통신하는 HTTP 클라이언트입니다. 로봇 생성, 세션 관리 등에 사용합니다. |
| `connect` | 로봇과 실시간 세션을 열고 `MenloSession`을 반환하는 고수준 함수입니다. |
| `generate_room_key` | *(실험 기능)* 브라우저 기반 3D 뷰어용 일회성 키를 생성합니다. |

연결 후에는 **`session`** 객체가 주요 핸들입니다.
- `session.discover_skills()` - 로봇이 할 수 있는 작업 목록 확인
- `session.invoke(skill, params)` - 스킬을 실행하고 결과 대기
- `session.get_vision(camera)` - 카메라 프레임 가져오기
- `session.state.get(key)` - 로봇의 현재 상태 읽기
- `session.disconnect()` - 연결 종료

**`robot_id`**는 플랫폼에서 로봇을 식별하는 고유 ID입니다. `client.robots.create()`를 호출하면 반환됩니다.


## 1단계: 의존성 설치

Colab에는 `menlo_robot_sdk`가 기본 설치되어 있지 않으므로 여기에서 설치합니다. 이 셀은 세션마다 한 번 실행해야 합니다. Colab 런타임이 초기화되면 설치된 패키지가 유지되지 않습니다.


In [ ]:
!pip install -q "menlo-robot-sdk[livekit]"

'C:\Python314\Scripts\pip.exe' ������ Device Guard ��å�� ���� ���ܵǾ����ϴ�.
�ڼ��� ������ ���� ����ڿ��� �����ϼ���.


## 2단계: 설정 불러오기

`google.colab.userdata`를 사용해 Colab Secrets 관리자에서 API 키를 불러옵니다.


In [ ]:
import os
import time
import asyncio

# ── 키 로더: 먼저 .env를 확인한 다음 Colab Secrets를 확인합니다 ────────────────────────
def _load_keys():
    # 모드 B: 로컬 .env 파일
    try:
        from dotenv import load_dotenv
        load_dotenv(override=False)
        if os.environ.get("MENLO_API_KEY"):
            print("Loaded keys from .env file")
            return "dotenv"
    except ImportError:
        pass  # python-dotenv가 설치되지 않았으므로 .env 모드가 아닙니다

    # 모드 A: Google Colab Secrets
    try:
        from google.colab import userdata
        os.environ.setdefault("MENLO_API_KEY",    userdata.get("MENLO_API_KEY") or "")
        os.environ.setdefault("TOKAMAK_API_KEY",  userdata.get("TOKAMAK_API_KEY") or "")
        print("Loaded keys from Colab Secrets")
        return "colab"
    except Exception:
        pass

    return "env"  # 키가 이미 환경 변수에 설정되어 있습니다(CI, Docker 등)

_load_keys()

MENLO_API_KEY   = os.environ.get("MENLO_API_KEY", "")
TOKAMAK_API_KEY = os.environ.get("TOKAMAK_API_KEY", "")
RCS_URL         = "https://api.menlo.ai/rcs"
VIEWER_BASE_URL = "https://sim.menlo.ai"

assert MENLO_API_KEY and not MENLO_API_KEY.startswith("sk_live_YOUR"), (
    "MENLO_API_KEY not set!\n"
    "  • Colab: add it in the Secrets manager (key icon in the left sidebar)\n"
    "  • Local: add MENLO_API_KEY=... to a .env file next to this notebook"
)
print(f"Configuration loaded — platform: {RCS_URL}")
print(f"MENLO_API_KEY    : {MENLO_API_KEY[:12]}...")
print(f"TOKAMAK_API_KEY  : {'(set)' if TOKAMAK_API_KEY else '(not set — needed for Section 7)'}")


## 짧은 설명: `await`는 무엇인가요?

이 튜토리얼의 거의 모든 SDK 호출은 `await` 키워드로 시작합니다. 처음 보는 학생을 위해 60초 버전으로 설명해 봅시다.

**문제:** 인터넷을 통해 로봇과 통신하려면 시간이 걸립니다. 플랫폼에 로봇 생성을 요청하거나 로봇에게 어디로 걸어가라고 말하면 답이 즉시 돌아오지 않습니다. 몇 밀리초에서 몇 초까지 걸릴 수 있습니다. 일반 Python 코드는 기다리는 동안 멈춘 것처럼 보입니다.

**해결:** Python의 *async*(비동기) 함수입니다. 비동기 함수는 네트워크 응답처럼 느린 작업을 기다리는 동안 전체 프로그램을 막지 않고 잠시 멈출 수 있습니다. `await` 키워드는 다음 뜻입니다.

> *"이 작업을 시작하고, 끝날 때까지 여기서 기다린 다음, 결과를 돌려줘."*

```python
# 일반 함수 호출 - 즉시 반환됩니다.
result = some_function()

# 비동기 호출 - 작업을 시작하고 완료될 때까지 기다립니다.
result = await some_async_function()
```

**기억할 규칙:** 플랫폼이나 로봇과 통신하는 모든 `menlo_robot_sdk` 호출은 비동기이므로 앞에 `await`가 필요합니다. 빼먹으면 데이터 대신 `<coroutine object ...>` 같은 값이 나오거나 `RuntimeWarning: coroutine was never awaited` 경고가 뜰 수 있습니다. 이 경고는 거의 항상 *`await`를 빼먹었다*는 뜻입니다.

**노트북 셀에서 왜 바로 `await`를 쓸 수 있나요?** 일반 Python에서는 `async def` 함수 안에서만 `await`를 쓸 수 있습니다. Jupyter와 Colab은 예외를 제공합니다. 노트북 셀은 이미 *이벤트 루프* 안에서 실행되므로 셀 최상위에서도 `await`를 사용할 수 있습니다. 이 코드를 일반 `.py` 스크립트로 옮기면 `async def main(): ...`으로 감싸고 `asyncio.run(main())`으로 실행해야 합니다.


## 3단계: 클라이언트와 로봇 만들기

`AsyncClient`는 RCS URL과 API 키를 사용해 Menlo 플랫폼에 연결합니다. 그런 다음 `asimov-v0` 창고 모델로 새 시뮬레이션 로봇을 만듭니다.


In [ ]:
from menlo_robot_sdk import AsyncClient

client = AsyncClient(rcs_url=RCS_URL, api_key=MENLO_API_KEY)

created = await client.robots.create(name=f"sdk-test-{int(time.time())}", model="asimov-v0")
robot_id = created.robot.id
print(f"Created robot: {robot_id}")

## 4단계: 세션 시작하기

세션을 열면 이 노트북이 로봇의 방에 참여하여 명령을 보낼 수 있습니다. 세션은 명시적으로 연결을 끊을 때까지 열린 상태로 유지됩니다.


In [ ]:
from menlo_robot_sdk import connect

session = await connect(
    client,
    robot_id,
    worker_names=[],                  # 서버 쪽 worker 없음 — 브라우저가 런타임입니다
    rcw_identity_prefix="simplesim",
    join_livekit=True,
)
print(f"Connected! room = {session.session.room_name}")


## 5단계: 3D 뷰어 열기

SDK가 일회성 뷰어 키를 생성하므로 브라우저에서 로봇을 볼 수 있습니다. **출력된 URL을 새 Chrome 탭에서 열고**, 3D 창고 장면이 완전히 로드될 때까지 기다린 뒤 계속하세요.

뷰어 링크는 4시간 동안 유효합니다. 회의 링크처럼 공유할 수 있습니다.


In [ ]:
from menlo_robot_sdk.experimental import generate_room_key

room_key = await generate_room_key(client, robot_id)
viewer_url = f"{VIEWER_BASE_URL}/?key={room_key}"

print("=" * 60)
print(f"VIEWER URL: {viewer_url}")
print("=" * 60)
print()
print("1. 위 URL을 새 Chrome 탭에서 여세요")
print("2. 3D 창고 로봇이 로드될 때까지 기다리세요")
print("3. 그런 다음 다음 셀을 실행하세요")


## 6단계: 스킬이 준비될 때까지 기다리기

스킬은 로봇이 수행할 수 있는 동작입니다. 스킬 탐색이 동작하려면 뷰어가 열려 있어야 합니다. 시뮬레이션이 브라우저에서 실행되기 때문입니다.

이 셀은 Chrome 탭이 방에 참여할 때까지 계속 재시도합니다.


In [ ]:
async def wait_for_skills(timeout_s: float = 180.0):
    """뷰어가 접속하고 스킬을 사용할 수 있을 때까지 주기적으로 확인합니다."""
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        try:
            found = await session.discover_skills()
            if found:
                return found
        except (RuntimeError, TimeoutError):
            pass  # 아직 뷰어가 접속하지 않았습니다
        await asyncio.sleep(2.0)
    raise TimeoutError("Viewer did not join — is the Chrome tab open?")


skills = await wait_for_skills()
print("Skills found:")
for skill in skills:
    print(f"  - {skill.name}")
    print(skill)


## 7단계: 로봇 상태 읽기

명령을 보내기 전에 로봇의 현재 상태를 확인해 봅시다. `robot_status`는 로봇의 위치, 방향, 전체 상태를 반환합니다.


In [ ]:
state = await session.state.get("robot_status")
print(f"Robot status : {state.robot.status}")
print(f"Robot entity : {state.robot.entity_id}")
if state.robot.pose:
    print(f"Position     : {state.robot.pose.position}")
    print(f"Yaw (deg)    : {state.robot.pose.yaw_deg}")
print(f"Runtime ready: {state.runtime.status}")
print(f"Accepts cmds : {state.runtime.accepts_tool_calls}")

## 8단계: 장면 상태 읽기

로봇 자신의 상태뿐 아니라 전체 **장면 상태**도 읽을 수 있습니다. 여기에는 창고 안의 모든 엔티티(패드, 큐브, 로봇)와 위치, 속성이 들어 있습니다.

장면 상태는 실제 로봇에서는 사용할 수 없습니다. 이는 미래의 실제 로봇 기능을 흉내 내기 위해 제공되는 시뮬레이션 전용 기능입니다. 스크립트가 장면 상태의 정보에 의존한다면, 그 스크립트는 실제 로봇으로 그대로 옮길 수 없다는 뜻입니다.

장면에는 세 종류의 엔티티가 있습니다.

| 엔티티 종류 | 예시 | 의미 |
|---|---|---|
| **로봇** | `robot` | 휴머노이드 자체입니다. 위치, 방향, 들고 있는 물체를 포함합니다. |
| **패드** | `pad_A`, `pad_B`, ... | 창고 바닥의 구역입니다. `pad_A`는 큐브를 공급하는 컨베이어 벨트이고, 나머지는 색상별 배송 패드입니다. 섹션 7에서 더 자세히 다룹니다. |
| **큐브** | `cube_0`, `cube_1`, ... | 컨베이어 위의 색깔 물체입니다. 집어서 배송할 수 있습니다. |

(+20, +20) 근처 멀리 보이지 않는 `visible=False` 상태의 `cube_pool_*` 엔티티도 보일 수 있습니다. 이는 시뮬레이터의 예비 큐브입니다. 큐브를 치우면 벨트가 이 숨겨진 풀에서 새 큐브를 가져오므로 창고의 큐브가 떨어지지 않습니다. `visible`로 필터링하면 숨길 수 있습니다.)


In [ ]:
scene = await session.state.get("scene_state")

# 엔티티를 종류별로 분류합니다
robot_entities = []
pad_entities = []
cube_entities = []

for eid, entity in scene.entities.items():
    if eid == "robot":
        robot_entities.append((eid, entity))
    elif eid.startswith("pad_"):
        pad_entities.append((eid, entity))
    elif eid.startswith("cube") and entity.visible:
        cube_entities.append((eid, entity))
    # 보이지 않는 예비 큐브와 중복 alias는 건너뜁니다

# --- 로봇 ---
print("=== Robot ===")
for eid, e in robot_entities:
    pos = e.pose.position
    print(f"  {eid:12s}  pos=({pos[0]:+.2f}, {pos[1]:+.2f}, {pos[2]:+.2f})  yaw={e.pose.yaw_deg:.1f}\u00b0")

# --- 패드 ---
print(f"\n=== Pads ({len(pad_entities)}) ===")
for eid, e in sorted(pad_entities):
    pos = e.pose.position
    print(f"  {eid:12s}  pos=({pos[0]:+.2f}, {pos[1]:+.2f}, {pos[2]:+.2f})")

# --- 보이는 큐브 ---
print(f"\n=== Cubes ({len(cube_entities)}) ===")
for eid, e in sorted(cube_entities):
    pos = e.pose.position
    color = e.state.get("color", "?") if e.state else "?"
    pad = e.state.get("parent_pad_id", "?") if e.state else "?"
    held = f"  [held by {e.attached_to}]" if e.attached_to else ""
    print(f"  {eid:12s}  color={color:6s}  on={pad:5s}  pos=({pos[0]:+.2f}, {pos[1]:+.2f}, {pos[2]:+.2f}){held}")


---

# 섹션 5: 휴머노이드 정책 뷰어 기본 사용법


## 뷰어 열기

아직 열지 않았다면 5단계에서 출력된 뷰어 URL을 새 Chrome 탭에서 여세요. 플랫폼 위에 서 있는 휴머노이드 로봇이 있는 3D 창고 장면이 보일 것입니다.

시뮬레이션은 실제로 이 뷰어에서 실행됩니다. 노트북은 네트워크로 명령을 보내고, 브라우저가 이를 실시간으로 실행합니다.


## 카메라 조작

3D 뷰어에서는 장면을 다양한 시점에서 볼 수 있도록 카메라를 움직일 수 있습니다.

| 동작 | 방법 |
|---|---|
| **회전** | 클릭 후 드래그(왼쪽 마우스 버튼) |
| **이동** | 오른쪽 클릭 후 드래그, 또는 Shift + 클릭 후 드래그 |
| **확대/축소** | 스크롤 휠 |
| **카메라 초기화** | `R` 키를 눌러 기본 시점으로 돌아가기 |

지금 카메라를 움직여 보세요. 창고 구조에 익숙해지는 것이 중요합니다. 로봇을 효과적으로 이동시키려면 물체들이 어디에 있는지 이해해야 합니다.


## Asimov 정책 뷰어

뷰어는 단순한 카메라가 아니라 `asimov-v0` 정책의 **런타임**입니다. 즉:

- 로봇의 물리, 충돌 감지, 내비게이션이 모두 **브라우저 탭 안에서** 실행됩니다.
- Chrome 탭을 닫으면 시뮬레이션이 멈추고 SDK가 런타임과의 연결을 잃습니다.
- 뷰어는 SDK 없이도 로봇을 직접 제어할 수 있는 여러 상호작용 모드를 제공합니다.

### 클릭해서 걷기

**지금 해보세요:** 창고 바닥 어딘가를 클릭하세요. 로봇이 클릭한 위치로 걸어갑니다.

가장 간단한 이동 방식입니다. 내부적으로는 SDK에서 사용할 `go_to` 스킬과 같은 기능이 실행됩니다.

### 자동 내비게이션

**지금 해보세요:** 라벨이 있는 패드 중 하나(`pad_A`, `pad_B`, `pad_C` 등)를 클릭하세요. 로봇이 경로를 계획하고 장애물을 피해 걸어갑니다.

로봇이 벽과 물체를 피하는 방식을 관찰하세요. 이 경로 계획 때문에 `go_to`가 원시 속도 명령보다 강력합니다.

### 수동 속도 명령

**지금 해보세요:** 방향키를 사용해 로봇을 수동으로 움직여 보세요.

| 키 | 동작 |
|---|---|
| `W` / `Up` | 앞으로 걷기 |
| `S` / `Down` | 뒤로 걷기 |
| `A` / `Left` | 왼쪽으로 돌기 |
| `D` / `Right` | 오른쪽으로 돌기 |

이는 SDK의 `set_velocity` 명령을 수동으로 사용하는 것과 같습니다. 키를 누르고 있는 동안 로봇이 움직입니다.


## 로봇이 넘어진 것을 어떻게 알 수 있나요?

뷰어에서는 로봇이 옆으로 눕거나 뒤로 넘어져 눈으로 쉽게 알 수 있습니다. 하지만 SDK에서는 코드로 확인해야 합니다.

`robot_status`에는 `status` 필드가 있습니다. 가능한 값은 다음과 같습니다.

| 상태 | 의미 |
|---|---|
| `ready` | 대기 중, 명령을 기다림 |
| `moving` | 이동 실행 중 |
| `holding` | 물체를 들고 있음 |
| **`fallen`** | **로봇이 넘어짐** |
| `error` | 문제가 발생함 |
| `unknown` | 아직 상태가 결정되지 않음 |

아래 셀을 실행해 확인하세요. `status`가 `fallen`이면 장면을 리셋해야 할 수 있습니다.


In [ ]:
state = await session.state.get("robot_status")
print(f"Robot status: {state.robot.status}")

if state.robot.status == "fallen":
    print("\n로봇이 넘어졌습니다! 장면을 리셋해야 할 수 있습니다.")
else:
    print("\n로봇이 서 있으며 정상적으로 동작할 수 있습니다.")


---

# 섹션 6: Menlo SDK로 간단한 명령 체인 작성하기


이 섹션에서는 SDK 명령을 연결한 짧은 프로그램을 작성합니다. 나중에 더 복잡한 로봇 행동을 만들 때도 이 방식이 기본이 됩니다.

노트북 안에 스크린샷을 찍고 표시하기 위한 헬퍼 함수를 사용합니다.


In [ ]:
from IPython.display import Image, display

async def screenshot(label: str = ""):
    """로봇의 POV 카메라 이미지를 가져와 노트북에 표시합니다."""
    jpeg = await session.get_vision("pov")
    if label:
        print(label)
    display(Image(data=jpeg, format="jpeg"))


## 연습 1 전에: `set_head`로 카메라 조준하기

로봇은 걷는 동작과 독립적으로 머리/목을 조준할 수 있습니다. 이렇게 하면 POV 카메라 방향이 바뀌므로, 어디로 이동할지 결정하기 전에 창고를 훑어보는 데 유용합니다.

`yaw`는 좌우 회전, `pitch`는 위아래 기울임입니다. 명령은 목표 각도를 설정하고, `robot_status`는 목표값과 실제 측정된 머리 상태를 모두 보고합니다.


In [ ]:
async def aim_head(yaw: float = 0.0, pitch: float = 0.0):
    """로봇의 머리/목을 조준한 뒤 목표값과 실제 측정값을 출력합니다."""
    result = await session.invoke("set_head", {"yaw": yaw, "pitch": pitch})
    print(f"set_head -> {result.status}")

    state = await session.state.get("robot_status")
    head = state.robot.extra.get("head", {})
    print("target  :", head.get("target"))
    print("measured:", head.get("measured"))
    print("ranges  :", head.get("yaw_range"), head.get("pitch_range"))


# 짧은 스캔을 시도합니다: 중앙, 약간 아래, 왼쪽, 오른쪽, 다시 중앙.
for label, yaw, pitch in [
    ("center", 0.0, 0.0),
    ("slightly down", 0.0, 0.15),
    ("left", 0.6, 0.15),
    ("right", -0.6, 0.15),
    ("center again", 0.0, 0.0),
]:
    print(f"\nHead pose: {label}")
    await aim_head(yaw=yaw, pitch=pitch)
    await screenshot(f"POV after aiming {label}:")


## 연습 1: 간단히 걷기

가장 기본적인 시퀀스를 해봅시다. 스크린샷을 찍고, 로봇을 앞으로 움직인 뒤, 다시 스크린샷을 찍습니다.

이를 통해 로봇의 시점에서 세상이 어떻게 보이는지, 움직인 뒤 어떻게 달라지는지 이해할 수 있습니다.


In [ ]:
# 1. 움직이기 전에 스크린샷을 찍습니다
await screenshot("BEFORE — robot's point of view:")


In [ ]:
# 2. 2초 동안 앞으로 이동합니다
result = await session.invoke("set_velocity", {"vx": 0.8, "duration_s": 2.0})
print(f"set_velocity -> {result.status}")


In [ ]:
# 3. 움직인 뒤 스크린샷을 찍습니다
await screenshot("AFTER — robot's point of view:")


### 생각해 볼 질문

위의 두 스크린샷을 보고 다음 질문을 생각해 보세요.

1. **시야가 바뀌었나요?**  
   전후 이미지를 자세히 비교하세요. 무엇이 달라졌나요?

2. **왜 시야가 바뀌었나요?**  
   로봇이 *얼마나 멀리* 이동했는지 추정할 수 있나요? 힌트: `set_velocity`에 넣은 값에서 속도 × 시간 = 거리입니다.

3. **로봇의 시점은 지금까지 보던 시점과 어떻게 다른가요?**  
   이제 세상을 보는 방법이 두 가지입니다. Chrome의 3D 뷰어와 로봇의 POV 카메라입니다. 한쪽에서는 보이지만 다른 쪽에서는 보이지 않는 것은 무엇인가요?

4. **로봇에 탑재된 센서의 한계를 고려하면 어떤 어려움이 예상되나요?**  
   POV 카메라만 사용할 수 있다고 상상해 보세요. 어떤 작업이 더 어려워질까요?


## 연습 2 전에: 특정 각도만큼 회전하는 법

연습 2에서는 로봇을 대략 90도 돌려야 합니다. 하지만 "90도 회전" 명령은 없습니다. `set_velocity`는 *회전 속도*와 *지속 시간*만 받습니다. 그래서 약간의 수학과 걷는 로봇의 특징을 이해해야 합니다.

### 도가 아니라 라디안

회전 속도 `wz`는 **초당 라디안**으로 측정됩니다. 라디안은 각도의 다른 단위입니다.

| 도 | 라디안(정확) | 라디안(근사) |
|---|---|---|
| 90도 | π/2 | 1.57 |
| 180도 | π | 3.14 |
| 360도 | 2π | 6.28 |

### 속도 × 시간 = 각도

걷기에서 거리 = 속도 × 시간인 것처럼 회전한 각도는 다음과 같습니다.

**회전 각도(라디안) = `wz` × `duration_s`**

### 특징 #1: 이 로봇은 자전거처럼 돕니다

`{"wz": 0.5, "duration_s": 3.14}`만 보내면 거의 돌지 않습니다. 다리를 움직이는 보행 정책은 **제자리 회전이 불가능**합니다. 자전거처럼 앞으로 움직이는 동안에만 잘 회전합니다. 항상 작은 전진 속도와 함께 사용하세요.

```python
{"wz": 0.5, "vx": 0.2, "duration_s": 3.14}   # 제대로 회전
{"wz": 0.5, "duration_s": 3.14}              # 거의 회전하지 않음
```

작은 `vx` 때문에 로봇은 제자리에서 피벗하는 대신 몇십 센티미터 정도의 완만한 호를 그립니다. 약간의 여유 공간을 계획하세요.

### 특징 #2: 속도 제한(조용한 클리핑)

정책은 특정 속도 범위까지만 훈련되었으므로, 시뮬레이터는 더 빠른 값을 **알려주지 않고 제한**합니다.

| 파라미터 | 의미 | 제한 |
|---|---|---|
| `vx` | 전진(+) / 후진(-), m/s | ±1.5 |
| `vy` | 왼쪽(+) / 오른쪽(-) 옆걸음, m/s | ±1.5 |
| `wz` | 왼쪽 회전(+) / 오른쪽 회전(-), rad/s | **±0.6** |

`wz = 1.0`을 요청해도 실제로는 0.6 rad/s로 회전하므로 계산한 각도가 틀어집니다. **수학과 실제 동작이 맞도록 `wz`는 0.6 이하로 유지하세요.**

### 재사용 가능한 레시피

```python
# 왼쪽으로 약 90도: 0.5 rad/s × 3.14 s ≈ 1.57 rad ≈ 90도
await session.invoke("set_velocity", {"wz": 0.5, "vx": 0.2, "duration_s": 3.14})

# 오른쪽으로 약 90도: 음수 wz는 반대 방향 회전
await session.invoke("set_velocity", {"wz": -0.5, "vx": 0.2, "duration_s": 3.14})

# 왼쪽으로 약 180도: 같은 속도, 두 배 시간
await session.invoke("set_velocity", {"wz": 0.5, "vx": 0.2, "duration_s": 6.28})
```

### 오차를 예상하세요

계산값은 목표이지 보장이 아닙니다. 실제 시뮬레이션에서 90도 명령은 대략 75도에서 100도 사이에 도착할 수 있습니다. 정책이 서서히 가속하고, 걸으면서 흔들리고, 즉시 멈추지 않기 때문입니다. 이후 실제 방향을 확인할 수 있습니다.

```python
state = await session.state.get("robot_status")
print(state.robot.pose.yaw_deg)   # degrees: 0도 = +x 방향, 90도 = +y 방향
```

실제 결과를 확인하고 보정하는 것은 로봇 공학의 중요한 아이디어입니다. 연습 2 끝에서 다시 돌아옵니다.


## 연습 2: 벽 돌아가기

이제 더 어려운 도전입니다. Chrome 뷰어의 3인칭 시점에서 창고 안의 벽이나 장애물을 찾으세요. 로봇 생성 지점 바로 앞의 컨베이어 벨트가 좋은 예입니다. 뒤쪽 공간으로 가는 직선 경로를 막고 있습니다.

**과제:** `go_to` 없이 `set_velocity` 명령만 사용해 장애물 주변을 돌아가는 스크립트를 작성하세요. `go_to`에는 경로 계획기가 내장되어 있어서 모든 일을 대신 해줍니다. 이 연습의 목적은 그런 기능이 없을 때 로봇을 제어하는 느낌을 경험하는 것입니다.

세 가지 속도 파라미터가 있습니다.

| 파라미터 | 의미 | 제한 |
|---|---|---|
| `vx` | 전진(+) / 후진(-) 속도(m/s) | ±1.5 |
| `vy` | 왼쪽(+) / 오른쪽(-) 옆걸음 속도(m/s) | ±1.5 |
| `wz` | 회전 속도(rad/s), 양수 = 왼쪽, 음수 = 오른쪽 | ±0.6 |

하나의 명령에서 조합할 수 있습니다. 특히 `wz`는 반드시 작은 `vx`와 함께 써야 합니다.

기본 전략은 다음과 같습니다.
1. 벽을 따라 바라보도록 약 90도 회전
2. 벽 끝을 지날 때까지 전진
3. 반대 방향으로 약 90도 회전
4. 전진하여 벽을 지난 위치로 이동
5. 필요하면 다시 회전하고 걸어서 벽 뒤쪽에 도착

회전하지 않고 `vy`로 옆걸음 하며 벽을 따라 이동해도 됩니다. 두 방법 모두 유효합니다. 어느 쪽이든 명령 사이마다 스크린샷과 로봇 위치를 확인하세요. 아래 골격 코드의 값을 채워 보세요.


In [ ]:
# 여러 번 재사용할 작은 헬퍼: 로봇의 위치와 방향을 출력합니다
async def print_position(label: str = ""):
    state = await session.state.get("robot_status")
    p = state.robot.pose.position
    print(f"{label:20s} pos=({p[0]:+.2f}, {p[1]:+.2f})  yaw={state.robot.pose.yaw_deg:+.1f}°  status={state.robot.status}")

# 현재 위치를 보기 위해 스크린샷을 찍습니다
await screenshot("Starting position:")
await print_position("START")


In [ ]:
# ============================================================
# 여기에 코드를 작성하세요
# set_velocity 명령을 여러 개 이어서 작성하여
# 로봇이 벽을 돌아가도록 만들어 보세요.
#
# 명령 예시:
#   await session.invoke("set_velocity", {"vx": 0.8, "duration_s": 3.0})               # 약 2m 전진
#   await session.invoke("set_velocity", {"wz": 0.5, "vx": 0.2, "duration_s": 3.14})   # 왼쪽으로 약 90° 회전
#   await session.invoke("set_velocity", {"wz": -0.5, "vx": 0.2, "duration_s": 3.14})  # 오른쪽으로 약 90° 회전
#
# 기억하세요:
#   * 회전에는 작은 vx가 필요합니다 — 이 로봇은 제자리 회전을 할 수 없습니다!
#   * |vx|, |vy| ≤ 1.5 and |wz| ≤ 0.6 — 더 빠른 값은 조용히 제한됩니다
#
# 팁: 이동 사이에 진행 상황을 확인하세요.
#   await screenshot("After turning:")
#   await print_position("after turn")
# ============================================================




In [ ]:
# 최종 위치를 확인합니다
await screenshot("Final position:")

state = await session.state.get("robot_status")
print(f"Final position: {state.robot.pose.position}")
print(f"Final yaw: {state.robot.pose.yaw_deg} degrees")
print(f"Robot status: {state.robot.status}")


### 생각해 볼 질문

스크립트가 동작했거나 몇 번 시도해 본 뒤 다음 질문을 생각해 보세요.

1. **스크립트가 매번 똑같이 동작하나요?**  
   2~3번 실행해 보세요. 먼저 로봇 위치를 리셋해야 할 수도 있습니다. 매번 정확히 같은 위치에 도착하나요? 왜 그럴까요?

2. **벽의 위치가 조금 바뀌면 어떻게 될까요?**  
   벽이 0.5m 정도 움직였다고 가정해 보세요. 스크립트가 여전히 동작할까요? 이 제어 방식의 한계는 무엇인가요?

3. **움직인 벽에 대응하려면 스크립트에 어떤 정보가 필요하고, 어디에서 얻을 수 있을까요?**  
   지금까지 사용한 도구를 떠올려 보세요. 로봇이 자신의 위치나 앞에 있는 물체를 알 수 있게 해주는 것이 있나요?


---

# 섹션 7: 집기와 놓기

걷는 것도 좋지만, 창고 로봇의 핵심은 *물건을 옮기는 것*입니다. 이 섹션에서는 로봇이 컨베이어 벨트의 큐브를 집어 패드로 배송합니다. 거의 모든 창고 로봇의 기본이 되는 고전적인 **pick and place** 작업입니다.

### 분류 게임

이 창고는 작은 게임이기도 합니다. 컨베이어(`pad_A`)는 색깔 큐브를 공급하고, 각 배송 패드는 **한 가지 색상만** 받습니다. 뷰어의 패드 표지판을 보세요. 색상으로 구분되어 있습니다.

| 색상 | 배송 패드 |
|---|---|
| green | `pad_C` |
| blue | `pad_D` |
| red | `pad_B` |
| yellow | `pad_E` |

큐브를 맞는 패드로 배송하면 창고가 큐브를 *수락*합니다. 큐브는 장면에서 사라지고 벨트가 새 큐브를 공급합니다.

> **주의:** 잘못된 색상의 패드로 배송하면 sorting benchmark가 종료됩니다. 이후 모든 `pick_entity`가 `benchmark terminated: wrong_color_destination` 오류로 실패합니다. 그런 경우 다음 명령으로 다시 준비시킬 수 있습니다.
> ```python
> await session.invoke("configure_benchmark", {"benchmark": {}})
> ```

### 새 스킬 두 가지

| 스킬 | 역할 | 사전 조건 |
|---|---|---|
| `pick_entity` | 큐브 집기 | 로봇이 큐브에 **닿을 만큼 가까워야** 함(약 1m), 손이 비어 있어야 함 |
| `place_entity` | 들고 있는 큐브를 패드로 배송 | 로봇이 뭔가를 들고 있어야 함 |

둘 다 `go_to`에서 본 것과 같은 **target 형식**을 사용합니다.

```python
{"target": {"kind": "entity", "entity_id": "<name>"}}
```

집는 방법은 두 가지입니다.
- **이름으로 집기** - `"entity_id": "cube_4"`는 정확히 그 큐브를 집습니다. 닿는 거리에 있어야 합니다.
- **가장 가까운 큐브 집기** - 특별한 이름 `"cube"`는 *가장 가까운 큐브*를 뜻합니다. 벨트 앞에서는 편리하지만, 큐브가 두 개 가까이 있으면 다른 것을 집을 수 있어 위험합니다.

어느 쪽이든 순서는 같습니다. **먼저 위치 잡기, 그다음 집기**입니다. 원하는 큐브 옆으로 걸어간 뒤 집으세요.


## 1단계: 장면 조사하기

먼저 지금 모든 것이 어디에 있는지 확인합니다. 섹션 4 이후 로봇은 움직였고, 컨베이어는 계속 큐브를 공급하고 있습니다. 정확한 위치를 위해 `scene_state`를 읽고, 로봇의 눈으로 보는 POV 스크린샷도 찍습니다.

참고: 실제 로봇에서는 장면 상태를 읽을 수 없습니다. 이 튜토리얼에서는 코드 동작을 단순하게 설명하기 위해 먼저 사용합니다.


In [ ]:
# 로봇과 큐브가 어디에 있는지 확인합니다
scene = await session.state.get("scene_state")

robot = scene.entities["robot"]
rp = robot.pose.position
print(f"Robot at ({rp[0]:+.2f}, {rp[1]:+.2f}), yaw={robot.pose.yaw_deg:.0f}°")

print("\nCubes (sorted by distance from the robot):")
cubes = []
for eid, e in scene.entities.items():
    if eid.startswith("cube_") and e.visible:
        p = e.pose.position
        dist = ((p[0] - rp[0]) ** 2 + (p[1] - rp[1]) ** 2) ** 0.5
        color = e.state.get("color", "?") if e.state else "?"
        cubes.append((dist, eid, color, p))

for dist, eid, color, p in sorted(cubes):
    print(f"  {eid:8s} color={color:6s} at ({p[0]:+.2f}, {p[1]:+.2f})  -> {dist:.2f} m away")

await screenshot("\nRobot's POV:")


## 2단계: 닿는 거리까지 이동하기

`pick_entity`는 로봇이 큐브에 충분히 가까이 서 있을 때만 동작합니다. 그래서 집기 전에 먼저 로봇을 큐브 옆에 **위치시킵니다**. 조사 결과에서 가장 가까운 큐브로 `go_to`를 사용해 이동합니다.

> **자주 빠지는 함정:** 로봇은 컨베이어 바로 앞에 정확히 생성되지 않습니다. 장면을 처음 로드하면 첫 번째 박스 옆에 조금 비켜서 있어 집기 범위 밖일 수 있습니다. 그래서 뷰어에서는 벨트가 가까워 보여도 첫 명령으로 `pick_entity`를 호출하면 실패합니다. "가까워 보인다"와 "닿는 거리 안에 있다"는 다릅니다. 그래서 눈대중 대신 먼저 위치를 잡고 `scene_state`로 거리를 측정합니다. 생성 직후에는 음수 `vy`로 약 1초 옆걸음 해도 해결되지만, 여기서는 일반적으로 동작하도록 `go_to`를 사용합니다.


In [ ]:
# 집을 수 있도록 가장 가까운 큐브까지 걸어갑니다
nearest_dist, nearest_cube, _, _ = min(cubes)
print(f"Nearest cube: {nearest_cube} ({nearest_dist:.2f} m away) — walking to it...")

result = await session.invoke(
    "go_to", {"target": {"kind": "entity", "entity_id": nearest_cube}}, timeout_s=300
)
print(f"go_to {nearest_cube} -> {result.status}")

# 이제 얼마나 가까워졌는지 확인합니다
scene = await session.state.get("scene_state")
rp = scene.entities["robot"].pose.position
cp = scene.entities[nearest_cube].pose.position
dist = ((cp[0] - rp[0]) ** 2 + (cp[1] - rp[1]) ** 2) ** 0.5
print(f"Distance to {nearest_cube} is now {dist:.2f} m")

await screenshot("In position:")


## 3단계: 집기!

이제 큐브 옆에 서 있으므로 집을 수 있습니다. 여기서는 `"cube"` target, 즉 *가장 가까운 큐브*를 사용하고, 바로 장면 상태를 확인해 무엇을 들고 있는지 확인합니다. 들린 큐브는 `attached_to` 필드가 설정되고 로봇 상태도 `holding`으로 바뀝니다.

왜 확인할까요? "가장 가까운"은 **집는 순간** 결정되고, 벨트는 계속 움직입니다. 한 큐브를 향해 걸어갔지만 이웃 큐브를 집을 수도 있습니다. 테스트 중 실제로 그런 일이 있었습니다. 4단계 코드는 실제로 집은 큐브의 색상을 읽고 맞는 패드로 보내므로 이런 상황을 안전하게 처리합니다. 특정 큐브가 꼭 필요하면 정확한 `entity_id`로 집으세요. 연습 문제에서 그렇게 해볼 것입니다.


In [ ]:
# 가장 가까운 큐브를 집습니다("cube" = 가장 가까운 큐브)
result = await session.invoke(
    "pick_entity", {"target": {"kind": "entity", "entity_id": "cube"}}, timeout_s=300
)
print(f"pick_entity -> {result.status}")

# 장면 상태로 실제 집은 큐브를 확인합니다: 어떤 큐브가 로봇에 붙어 있나요?
scene = await session.state.get("scene_state")
held = [
    (eid, e.state.get("color", "?") if e.state else "?")
    for eid, e in scene.entities.items()
    if eid.startswith("cube_") and e.attached_to
]
print(f"Now holding: {held}")

state = await session.state.get("robot_status")
print(f"Robot status: {state.robot.status}")   # 'holding'이어야 합니다

await screenshot("After picking:")


## 4단계: 맞는 패드로 운반하고 배송하기

로봇은 큐브를 들고 걸을 때 자동으로 운반합니다. 하지만 분류 규칙을 기억하세요. 큐브는 **색상과 맞는 패드**로 가야 하며, 틀리면 benchmark가 종료됩니다. 아래 코드는 다음을 수행합니다.

1. 실제로 들고 있는 큐브의 **색상**을 확인합니다(`scene_state`).
2. 작은 딕셔너리에서 맞는 패드를 찾습니다.
3. `go_to`(경로 계획기)로 그곳까지 걸어갑니다.
4. `place_entity`로 배송합니다.
5. **배송을 검증**합니다. 수락된 큐브는 장면에서 사라지고(`visible`이 `False`), 로봇은 `ready`로 돌아갑니다.

5단계가 흥미로운 이유는 성공하면 큐브가 *사라지기* 때문입니다. 눈으로 볼 대상이 없어지므로, 세계 상태를 읽는 것이 유일한 확실한 확인 방법입니다.

추가로, `place_entity`의 target은 생략할 수 있습니다. 생략하면 로봇은 **가장 가까운** 알려진 구역에 배송합니다. 하지만 "가장 가까움"과 색상 규칙을 섞으면 사고가 나기 쉬우므로, 우리는 항상 패드를 명시합니다.


In [ ]:
# 어떤 패드가 어떤 색상을 받는지 정의합니다(뷰어의 패드 표지판도 확인하세요!)
COLOR_TO_PAD = {"green": "pad_C", "blue": "pad_D", "red": "pad_B", "yellow": "pad_E"}

# 1-2. 무엇을 들고 있으며 어디로 가야 하나요?
scene = await session.state.get("scene_state")
held_cube, held_color = next(
    (eid, e.state.get("color"))
    for eid, e in scene.entities.items()
    if eid.startswith("cube_") and e.attached_to
)
target_pad = COLOR_TO_PAD[held_color]
print(f"Holding {held_cube} ({held_color}) -> delivering to {target_pad}")

# 3. 경로 계획기로 그곳까지 걸어갑니다
result = await session.invoke(
    "go_to", {"target": {"kind": "entity", "entity_id": target_pad}}, timeout_s=300
)
print(f"go_to {target_pad} -> {result.status}")

# 4. 배송합니다
result = await session.invoke(
    "place_entity", {"target": {"kind": "entity", "entity_id": target_pad}}, timeout_s=300
)
print(f"place_entity -> {result.status}")

# 5. 눈이 아니라 데이터로 검증합니다: 배송된 큐브는 사라져야 하며,
#    로봇은 더 이상 아무것도 들고 있지 않아야 합니다.
scene = await session.state.get("scene_state")
state = await session.state.get("robot_status")
print(f"{held_cube} still visible? {scene.entities[held_cube].visible}")
print(f"Robot status: {state.robot.status}")          # 다시 'ready'
delivered = not scene.entities[held_cube].visible and state.robot.status != "holding"
print(f"Delivery confirmed: {delivered}")

await screenshot("After delivering:")


### 생각해 볼 질문

1. **로봇이 모든 큐브에서 멀리 떨어져 있을 때 `pick_entity`를 호출하면 어떻게 될까요?** 예측한 뒤 빈 공간으로 걸어가서 시도해 보세요. `result`는 어떻게 보이나요?

2. **이미 큐브를 들고 있을 때 다시 `pick_entity`를 호출하면 어떻게 될까요?** 섹션 4, 6단계의 스킬 설명을 다시 보세요.

3. **큐브를 잘못된 색상의 패드로 배송하면 어떻게 될까요?** 실제로 시도하면 영향이 있으니, 이 섹션 위쪽의 경고를 먼저 읽으세요.

4. **배송 후 뷰어만 보거나 `result.status`만 믿지 않고, 큐브의 `visible` 플래그와 로봇 상태를 확인한 이유는 무엇일까요?**


## 연습: 빨간 큐브 배송하기

이제 여러분 차례입니다. **컨베이어에서 *빨간* 큐브를 찾아 올바른 패드로 배송하는 프로그램**을 작성하세요.

이 섹션의 모든 내용을 결합합니다.

- `scene_state`로 빨간 큐브를 *찾으세요*. 큐브 이름을 하드코딩하지 마세요. 벨트가 움직이고 다시 채워지므로 실행마다 ID가 바뀔 수 있습니다.
- 그 큐브로 걸어가서 집고, **실제로 들고 있는 것이 무엇인지 확인**하세요. 정말 빨간색인가요? `"cube"` nearest-pick을 사용했다면 어떤 문제가 생길 수 있나요?
- 올바른 패드로 배송하세요. 이 섹션 위쪽 표를 보세요. 틀리면 benchmark가 종료됩니다.
- 배송이 수락되었는지 **코드로 검증**하세요. 배송된 큐브는 어떻게 되나요?

주의할 점: 로봇이 두 큐브 사이에 멈추면 "가장 가까운" 큐브가 원하는 큐브가 아닐 수 있습니다. 코드는 이를 어떻게 알아차릴 수 있고, 더 안전한 집기 방법은 무엇일까요?


In [ ]:
# ============================================================
# 여기에 코드를 작성하세요
#
# 계획:
#   1. scene_state를 읽고 빨간 큐브를 찾습니다(e.state["color"] 확인)
#   2. 해당 큐브까지 go_to로 이동합니다
#   3. pick_entity를 정확한 entity_id로 호출합니다("cube"보다 안전합니다!)
#   4. 실제로 들고 있는 것(attached_to)이 빨간 큐브인지 확인합니다
#   5. 빨간색을 받는 패드로 go_to한 뒤 place_entity를 호출합니다
#   6. scene_state를 다시 읽어 배송을 증명합니다:
#      들고 있던 큐브의 visible이 False가 되어야 합니다
#
# 유용한 코드 조각:
#   await session.invoke("go_to", {"target": {"kind": "entity", "entity_id": "..."}}, timeout_s=300)
#   color = e.state.get("color") if e.state else None
# ============================================================




---
## 정리


In [ ]:
await session.disconnect()
await client.robots.delete(robot_id)
print("Robot deleted.")